# GPflow syntax: GPR
This is a simple notebook to show how to run a standard GPR model.

## Setup & Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

In [ ]:
import matplotlib
import numpy as np
import tensorflow as tf

In [ ]:
import gpflow
# from gpflow.ci_utils import ci_niter
from gpflow.utilities import print_summary

In [ ]:
sys.path.append('../../FCEst-benchmarking')

from helpers.synthetic_covariance_structures import get_constant_covariances, get_periodic_covariances, get_stepwise_covariances
from helpers.synthetic_covariance_structures import get_d2_covariance_structure
from helpers.simulations import simulate_time_series

In [ ]:
%matplotlib inline
matplotlib.rcParams['figure.figsize'] = (12, 6)
plt = matplotlib.pyplot

## Generate bivariate data

In [ ]:
N = 400

In [ ]:
cov_structure = get_d2_covariance_structure(
    get_periodic_covariances(num_samples=N, num_periods=1)
)
cov_structure.shape

In [ ]:
x = np.linspace(0, 1, N).reshape(-1, 1)
x.shape

In [ ]:
ts = cov_structure[:, 0, 1].reshape(-1, 1)
ts.shape

In [ ]:
plt.figure()
plt.plot(x, ts, 'x-')
plt.xlabel('time [steps]')

# Gaussian Process Regression
This is an example to see the syntax of GPflow.

In [ ]:
k = gpflow.kernels.Matern52()
print_summary(k)

In [ ]:
# GPflow will automatically assign a Gaussian likelihood
m = gpflow.models.GPR(
    data=(x, ts),
    kernel=k, 
    mean_function=None
)

In [ ]:
m.likelihood.variance.assign(0.01)
m.kernel.lengthscales.assign(0.3)

In [ ]:
print_summary(m)

## Optimization

In [ ]:
opt = gpflow.optimizers.Scipy()

In [ ]:
m.trainable_variables

In [ ]:
def objective_closure():
    return -m.log_marginal_likelihood()

In [ ]:
opt_logs = opt.minimize(
    objective_closure, 
    m.trainable_variables, 
    options=dict(maxiter=100)
)

In [ ]:
print_summary(m)

## Predictions

In [ ]:
# predict mean and variance of latent GP at test points
mean, var = m.predict_f(x)
mean.shape

In [ ]:
plt.figure()
plt.plot(x, ts, 'kx')
plt.plot(x, mean, "C0", lw=2)
plt.fill_between(
    x[:, 0],
    mean[:, 0] - 1.96 * np.sqrt(var[:, 0]),
    mean[:, 0] + 1.96 * np.sqrt(var[:, 0]),
    color="C0",
    alpha=0.2,
)
plt.xlabel('time [steps]')

In [ ]:
# generate 10 samples from posterior
tf.random.set_seed(1)  # for reproducibility
samples = m.predict_f_samples(x, 3)
samples.shape

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(x, ts, "k")
_ = plt.plot(x, samples[:, :, 0].numpy().T, "C0", linewidth=0.5)